In [ ]:
from huggingface_hub import login, HfApi
from dotenv import load_dotenv

# 1. 환경 변수 로드
load_dotenv("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/ASAN_01_mini_SAM3/.env")
hf_token = os.getenv("HUGGINGFACE_TOKEN")

if hf_token is None:
    raise ValueError("HUGGINGFACE_TOKEN을 env 파일에서 찾을 수 없음")

login(token=hf_token)

: 

In [7]:
import os
import sys
import torch
import json
import numpy as np
import cv2
import random
import glob
from PIL import Image
from typing import List, Dict, Any
from collections import OrderedDict

# --- SAM3 라이브러리 ---
import sam3
from sam3 import build_sam3_image_model
from sam3.model_builder import build_sam3_video_model
from sam3.train.data.collator import collate_fn_api as collate
from sam3.model.utils.misc import copy_data_to_device
from sam3.train.data.sam3_image_dataset import InferenceMetadata, FindQueryLoaded, Image as SAMImage, Datapoint
from sam3.train.transforms.basic_for_api import ComposeAPI, RandomResizeAPI, ToTensorAPI, NormalizeAPI
from sam3.eval.postprocessors import PostProcessImage

# sam3_root 경로 설정 (환경에 맞게 수정 필요)
sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")
sys.path.append(f"{sam3_root}/examples")

# --- 설정 ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GLOBAL_COUNTER = 1

# ==============================================================================
# [Helper] 마스크 인코딩 (RLE)
# ==============================================================================
def mask_to_rle(mask):
    """
    이진 마스크(Binary Mask)를 RLE(Run-Length Encoding) 형식으로 변환합니다.
    JSON 저장 시 용량을 획기적으로 줄여줍니다.
    """
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return {"size": mask.shape, "counts": runs.tolist()}

# ==============================================================================
# [Part 1] 이미지 내 객체 검출 (Detection) 관련 함수
# ==============================================================================

def create_empty_datapoint():
    return Datapoint(find_queries=[], images=[])

def set_image(datapoint, pil_image):
    w, h = pil_image.size
    datapoint.images = [SAMImage(data=pil_image, objects=[], size=[h, w])]

def add_text_prompt(datapoint, text_query):
    global GLOBAL_COUNTER
    assert len(datapoint.images) == 1, "이미지를 먼저 설정해주세요."
    w, h = datapoint.images[0].size
    datapoint.find_queries.append(
        FindQueryLoaded(
            query_text=text_query,
            image_id=0,
            object_ids_output=[],
            is_exhaustive=True,
            query_processing_order=0,
            inference_metadata=InferenceMetadata(
                coco_image_id=GLOBAL_COUNTER,
                original_image_id=GLOBAL_COUNTER,
                original_category_id=1,
                original_size=[w, h],
                object_id=0,
                frame_index=0,
            )
        )
    )
    GLOBAL_COUNTER += 1
    return GLOBAL_COUNTER - 1

def detect_objects_in_first_frame(
    video_dir: str, 
    text_prompt: str, 
    model_checkpoint_path: str = None
) -> Dict[str, Any]:
    """
    1. 이미지 프레임 폴더와 프롬프트를 입력받아 첫 번째 프레임에서 객체를 찾습니다.
    """
    print("\n" + "="*60)
    print(f" [Step 1] 객체 검출 (Detection) 시작: '{text_prompt}' ")
    print("="*60)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. 첫 번째 프레임 경로 찾기
    candidates = sorted(glob.glob(os.path.join(video_dir, "*.jpg")))
    if not candidates:
        print(f"[Error] 폴더 내에 jpg 이미지가 없습니다: {video_dir}")
        return None, None
    
    img_path = candidates[0]
    print(f"[Info] 첫 번째 프레임 로드: {img_path}")

    # 2. 이미지 모델 로드
    print("[Model] SAM 3 Image Model 로드 중...")
    if model_checkpoint_path is None:
        # 기본 체크포인트 경로
        model_checkpoint_path = "/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/checkpoints/SAM3/bpe_simple_vocab_16e6.txt.gz"
    
    model = build_sam3_image_model(bpe_path=model_checkpoint_path)
    model.to(device)

    # 3. 전처리 및 데이터셋 구성
    transform = ComposeAPI(
        transforms=[
            RandomResizeAPI(sizes=1008, max_size=1008, square=True, consistent_transform=False),
            ToTensorAPI(),
            NormalizeAPI(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ]
    )
    postprocessor = PostProcessImage(
        max_dets_per_img=-1,
        iou_type="segm",
        use_original_sizes_box=True,
        use_original_sizes_mask=True,
        convert_mask_to_rle=False,
        detection_threshold=0.5,
        to_cpu=False, # GPU 텐서 상태로 유지
    )

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        img_pil = Image.open(img_path).convert("RGB")
        datapoint = create_empty_datapoint()
        set_image(datapoint, img_pil)
        
        print(f"[Prompt] 텍스트 프롬프트 적용: '{text_prompt}'")
        add_text_prompt(datapoint, text_prompt)
        
        datapoint = transform(datapoint)

        # 4. 추론 수행
        print("[Inference] 추론 시작...")
        batch = collate([datapoint], dict_key="dummy")["dummy"]
        batch = copy_data_to_device(batch, device, non_blocking=True)
        
        output = model(batch)
        processed_results = postprocessor.process_results(output, batch.find_metadatas)

    # 5. 결과 추출
    if len(processed_results) > 0:
        first_result = list(processed_results.values())[0]
        num_obj = first_result["scores"].numel()
        print(f"[Result] 검출 완료: {num_obj}개의 객체 발견.")
        
        # 모델 메모리 해제
        del model; del output; del batch
        torch.cuda.empty_cache()
        
        return first_result, img_path
    else:
        print("[Result] 검출된 결과가 없습니다.")
        return None, img_path


# ==============================================================================
# [Part 2] 비디오 트래킹 (Tracking) 관련 함수 및 클래스
# ==============================================================================

class LazyVideoLoader:
    """이미지를 필요할 때만 디스크에서 읽어오는 로더 (메모리 절약)"""
    def __init__(self, video_path, image_size=1008):
        self.video_path = video_path
        self.image_size = image_size
        self.frame_paths = sorted(glob.glob(os.path.join(video_path, "*.jpg")) + 
                                  glob.glob(os.path.join(video_path, "*.jpeg")) +
                                  glob.glob(os.path.join(video_path, "*.png")))
        try:
            self.frame_paths.sort(key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
        except:
            self.frame_paths.sort()
        print(f"[LazyLoader] 총 {len(self.frame_paths)}개의 프레임 준비됨.")

    def __len__(self):
        return len(self.frame_paths)

    def __getitem__(self, idx):
        img_path = self.frame_paths[idx]
        img = cv2.imread(img_path)
        if img is None:
            raise RuntimeError(f"이미지 로드 실패: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # SAM3 전처리
        img = cv2.resize(img, (self.image_size, self.image_size))
        img = img.astype(np.float32) / 255.0
        img -= np.array([0.485, 0.456, 0.406])
        img /= np.array([0.229, 0.224, 0.225])
        
        return torch.from_numpy(img).permute(2, 0, 1)

def init_state_lazy(predictor, video_path, offload_state_to_cpu=True):
    """LazyLoader를 사용하는 커스텀 init_state 함수"""
    inference_state = {}
    inference_state["offload_video_to_cpu"] = True
    inference_state["offload_state_to_cpu"] = offload_state_to_cpu
    inference_state["device"] = predictor.device
    inference_state["storage_device"] = torch.device("cpu") if offload_state_to_cpu else torch.device("cuda")

    inference_state["images"] = LazyVideoLoader(video_path, image_size=predictor.image_size)
    inference_state["num_frames"] = len(inference_state["images"])
    
    first_img = cv2.imread(inference_state["images"].frame_paths[0])
    inference_state["video_height"] = first_img.shape[0]
    inference_state["video_width"] = first_img.shape[1]
    
    # 기본 구조 초기화
    inference_state["point_inputs_per_obj"] = {}
    inference_state["mask_inputs_per_obj"] = {}
    inference_state["cached_features"] = {}
    inference_state["constants"] = {}
    inference_state["obj_id_to_idx"] = OrderedDict()
    inference_state["obj_idx_to_id"] = OrderedDict()
    inference_state["obj_ids"] = []
    inference_state["output_dict"] = {
        "cond_frame_outputs": {},
        "non_cond_frame_outputs": {},
    }
    inference_state["first_ann_frame_idx"] = None
    inference_state["output_dict_per_obj"] = {}
    inference_state["temp_output_dict_per_obj"] = {}
    inference_state["consolidated_frame_inds"] = {
        "cond_frame_outputs": set(),
        "non_cond_frame_outputs": set(),
    }
    inference_state["tracking_has_started"] = False
    inference_state["frames_already_tracked"] = {}
    
    predictor.clear_all_points_in_video(inference_state)
    return inference_state

def track_objects_in_video(
    image_path: str, 
    detection_results: Dict[str, Any], 
    output_dir: str
):
    print("\n" + "="*60)
    print(" [Step 2] 비디오 트래킹 및 프레임별 JSON 저장 시작 ")
    print("="*60)

    video_dir = os.path.dirname(image_path)
    print(f"[Info] 비디오 소스 경로: {video_dir}")

    # 마스크 키 확인
    mask_key = "masks" if "masks" in detection_results else "segmentation"
    if mask_key not in detection_results:
        print(f"[Error] 마스크 정보 없음: {detection_results.keys()}")
        return

    num_detected = detection_results["scores"].numel()
    if num_detected == 0:
        print("[Error] 추적할 객체가 없습니다.")
        return
    
    print(f"[Info] 총 {num_detected}개의 객체 추적 시작.")

    # 1. 모델 로드
    print("[Model] SAM 3 Video Model (Tracker) 로드 중...")
    sam3_model = build_sam3_video_model(apply_temporal_disambiguation=True, device="cuda")
    predictor = sam3_model.tracker
    predictor.backbone = sam3_model.detector.backbone
    print("[Model] 로드 완료.")

    # 2. Lazy 세션 초기화
    try:
        inference_state = init_state_lazy(predictor, video_dir, offload_state_to_cpu=True)
    except Exception as e:
        print(f"[Error] 세션 초기화 실패: {e}")
        return

    # 3. 마스크 등록 (Multi-Object)
    print(f"[Tracking] {num_detected}개 객체 등록 중...")
    obj_colors = {}
    for i in range(num_detected):
        obj_colors[i+1] = [random.randint(50, 255) for _ in range(3)]

    for i in range(num_detected):
        mask = detection_results[mask_key][i]
        mask_input = mask.cuda().float()

        # [중요] 3차원(1, H, W)인 경우 2차원(H, W)으로 변경 (AssertionError 방지)
        if mask_input.dim() == 3:
            mask_input = mask_input.squeeze(0)
        
        score = detection_results["scores"][i].item()
        obj_id = i + 1

        predictor.add_new_mask(
            inference_state=inference_state,
            frame_idx=0,
            obj_id=obj_id,
            mask=mask_input
        )
        print(f"  - ID:{obj_id} 등록 (Score: {score:.4f})")

    # 4. 저장 폴더 생성
    print("[Tracking] 전파(Propagation) 시작...")
    
    # 이미지용 폴더
    vis_output_dir = os.path.join(output_dir, "tracking_images")
    os.makedirs(vis_output_dir, exist_ok=True)
    
    # JSON용 폴더
    json_output_dir = os.path.join(output_dir, "tracking_jsons")
    os.makedirs(json_output_dir, exist_ok=True)

    # 5. 트래킹 루프
    for frame_idx, obj_ids, low_res_masks, video_res_masks, obj_scores in predictor.propagate_in_video(
        inference_state, 
        start_frame_idx=0, 
        max_frame_num_to_track=None, 
        reverse=False, 
        propagate_preflight=True 
    ):
        # [메모리 관리] 100프레임마다 오래된 'non_cond' 데이터 삭제
        if frame_idx % 100 == 0:
            print(f"  > Frame {frame_idx}/{inference_state['num_frames']} Processing... (VRAM Cleanup)")
            cutoff = frame_idx - 50
            if cutoff > 0:
                outputs = inference_state["output_dict"]
                # non_cond_frame_outputs 만 삭제 (0번 프레임 등 cond_frame은 보존)
                keys_to_remove = [k for k in outputs["non_cond_frame_outputs"] if k < cutoff]
                for k in keys_to_remove:
                    del outputs["non_cond_frame_outputs"][k]
                
                for obj_dict in inference_state["output_dict_per_obj"].values():
                    keys_to_remove = [k for k in obj_dict["non_cond_frame_outputs"] if k < cutoff]
                    for k in keys_to_remove:
                        del obj_dict["non_cond_frame_outputs"][k]
                        
                if "cached_features" in inference_state:
                    cache_keys = [k for k in inference_state["cached_features"] if k < cutoff]
                    for k in cache_keys:
                        del inference_state["cached_features"][k]

        # --- 데이터 구성 ---
        frame_path = inference_state["images"].frame_paths[frame_idx]
        file_name = os.path.basename(frame_path)
        
        frame_results = {
            "frame_index": frame_idx,
            "file_name": file_name,
            "objects": []
        }

        # 시각화 및 데이터 추출
        frame_img = cv2.imread(frame_path)
        
        if frame_img is not None and video_res_masks is not None and len(video_res_masks) > 0:
            for k, obj_id in enumerate(obj_ids):
                if isinstance(obj_id, torch.Tensor): obj_id = obj_id.item()
                mask_tensor = video_res_masks[k]
                if mask_tensor.dim() == 3: mask_tensor = mask_tensor.squeeze(0)
                
                # Binary Mask
                mask_np = (mask_tensor.cpu().numpy() > 0.0).astype(np.uint8)
                
                if np.any(mask_np):
                    # 1. JSON용 RLE 저장
                    rle = mask_to_rle(mask_np)
                    frame_results["objects"].append({
                        "id": obj_id,
                        "segmentation": rle
                    })
                    
                    # 2. 시각화 (이미지 저장용)
                    color = obj_colors.get(obj_id, [0, 0, 255])
                    ys, xs = np.where(mask_np)
                    if len(ys) > 0:
                        alpha = 0.5
                        roi = frame_img[ys, xs]
                        blended = (roi.astype(float) * (1 - alpha) + np.array(color) * alpha).astype(np.uint8)
                        frame_img[ys, xs] = blended

            # 이미지 파일 저장
            cv2.imwrite(os.path.join(vis_output_dir, f"{frame_idx:06d}.jpg"), frame_img)

        # JSON 파일 개별 저장 (프레임별)
        json_filename = f"{frame_idx:06d}.json"
        json_path = os.path.join(json_output_dir, json_filename)
        try:
            with open(json_path, 'w') as f:
                json.dump(frame_results, f)
        except Exception as e:
            print(f"[Warning] Frame {frame_idx} JSON 저장 실패: {e}")

    print(f"[Success] 트래킹 완료.")
    print(f"  - 이미지 경로: {vis_output_dir}")
    print(f"  - JSON 경로: {json_output_dir}")

    # 정리
    del predictor; del sam3_model; del inference_state
    torch.cuda.empty_cache()


# ==============================================================================
# [Main] 메인 실행 함수
# ==============================================================================
def main():
    # 1. 설정
    video_dir = "/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/1_FRAME/AI_dataset/N01/N01_Ward/frontal__knee_flexion"
    prompt = "person"
    # 절대 경로 사용 (권한 에러 방지)
    output_dir = "/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/test/sam3"
    
    os.makedirs(output_dir, exist_ok=True)

    # 2. [함수 1] 객체 검출 실행
    detection_results, first_img_path = detect_objects_in_first_frame(
        video_dir=video_dir,
        text_prompt=prompt
    )

    if detection_results is None:
        print("[Main] 검출 실패로 종료합니다.")
        return

    # 3. 중간 메모리 정리
    print("[Main] Detection 완료. VRAM 정리를 수행합니다.")
    torch.cuda.empty_cache()

    # 4. [함수 2] 비디오 트래킹 실행
    track_objects_in_video(
        image_path=first_img_path,
        detection_results=detection_results,
        output_dir=output_dir
    )

if __name__ == "__main__":
    main()


 [Step 1] 객체 검출 (Detection) 시작: 'person' 
[Info] 첫 번째 프레임 로드: /workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/1_FRAME/AI_dataset/N01/N01_Ward/frontal__knee_flexion/000000.jpg
[Model] SAM 3 Image Model 로드 중...
[Prompt] 텍스트 프롬프트 적용: 'person'
[Inference] 추론 시작...
[Result] 검출 완료: 1개의 객체 발견.
[Main] Detection 완료. VRAM 정리를 수행합니다.

 [Step 2] 비디오 트래킹 및 프레임별 JSON 저장 시작 
[Info] 비디오 소스 경로: /workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/1_FRAME/AI_dataset/N01/N01_Ward/frontal__knee_flexion
[Info] 총 1개의 객체 추적 시작.
[Model] SAM 3 Video Model (Tracker) 로드 중...


INFO 2026-01-20 16:57:25,751 252 sam3_video_base.py: 125: setting max_num_objects=10000 and num_obj_for_compile=16


[Model] 로드 완료.
[LazyLoader] 총 331개의 프레임 준비됨.
[Tracking] 1개 객체 등록 중...
  - ID:1 등록 (Score: 0.9414)
[Tracking] 전파(Propagation) 시작...


propagate in video:   0% 0/331 [00:00<?, ?it/s]

  > Frame 0/331 Processing... (VRAM Cleanup)


...

  > Frame 200/331 Processing... (VRAM Cleanup)


propagate in video:  91% 301/331 [01:05<00:06,  4.33it/s]

  > Frame 300/331 Processing... (VRAM Cleanup)


propagate in video: 100% 331/331 [01:12<00:00,  4.58it/s]

[Success] 트래킹 완료.
  - 이미지 경로: /workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/test/sam3/tracking_images
  - JSON 경로: /workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/test/sam3/tracking_jsons
